# 🌱 AgriGuard — AI & Data Demo Notebook
**Person 2: AI & Data Developer**

AgriGuard is an AI-powered pest detection and community outbreak early-warning
system for Rwandan smallholder farmers. This notebook demonstrates the **complete
AI and data workflow** in Google Colab:

| Section | Topic |
|---------|-------|
| 1 | Install dependencies |
| 2 | Import libraries |
| 3 | Load data |
| 4 | Explore the data |
| 5 | Preprocess data |
| 6 | Crop / problem analysis |
| 7 | Confidence calculation |
| 8 | Severity assessment |
| 9 | Knowledge retrieval (RAG) |
| 10 | AI recommendation |
| 11 | Community risk analysis |
| 12 | Outbreak detection |
| 13 | 5 km alert logic |
| 14 | Agronomist escalation |
| 15 | Multilingual example |
| 16 | End-to-end demonstration (Scenarios A–D) |

---
**PROTOTYPE DATA NOTICE** All farmer reports and location data in this
notebook are simulated demo data. They do NOT represent real current disease
outbreaks. See Section 3 for the full honesty declaration.

**AI DISCLAIMER** This prototype AI assessment does NOT replace professional
agronomist advice.

## ⬇️  Cell 0 — Clone the repository (run this FIRST in Google Colab)

This cell clones the AgriGuard repository from GitHub so the `ai/src/` modules
and `ai/data/` CSV files are available. **Run this cell once before anything else.**

If you already have the repo (e.g. uploaded manually or mounted from Drive),
you can skip it — just make sure `REPO_ROOT` points to the repo root.

In [ ]:
import os, sys

# ── Clone from GitHub (only needed in Colab) ─────────────────────────────
REPO_URL  = 'https://github.com/Klab-TekHer-Cohort2/AGRIGUARD.git'
CLONE_DIR = '/content/AGRIGUARD'

if not os.path.exists(CLONE_DIR):
    print('Cloning AgriGuard repository...')
    os.system(f'git clone --depth 1 {REPO_URL} {CLONE_DIR}')
    print('Clone complete.')
else:
    print('Repository already present — pulling latest changes...')
    os.system(f'git -C {CLONE_DIR} pull --ff-only')

# Add the repo root to Python path so `from ai.src import ...` works
REPO_ROOT = CLONE_DIR
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Point the data loader at the cloned data directory
os.environ['AGRIGUARD_DATA_DIR'] = os.path.join(REPO_ROOT, 'ai', 'data')

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'Data dir  : {os.environ["AGRIGUARD_DATA_DIR"]}')
print('Ready.')

## Section 1 — Install Dependencies
Run this cell next. In Google Colab you can click **Runtime → Run all** to run
every cell in order (Cell 0 must finish first).

Core dependencies (pandas, numpy, scikit-learn, matplotlib) are lightweight and
always needed. The optional `sentence-transformers` and `faiss-cpu` enable
semantic RAG with real embeddings; the pipeline falls back to keyword retrieval
if they are absent.

In [ ]:
# ── Install dependencies (Colab) ──────────────────────────────────────────
import sys, subprocess

CORE = [
    'pandas>=2.0', 'numpy>=1.24', 'scikit-learn>=1.3',
    'matplotlib>=3.7', 'Pillow>=10.0', 'python-dotenv>=1.0',
]
OPTIONAL = [
    'sentence-transformers>=2.7',
    'faiss-cpu>=1.7',
]

print('Installing core dependencies...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + CORE)
print('Core ✓')

print('Installing optional semantic RAG dependencies...')
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + OPTIONAL)
    print('sentence-transformers + faiss-cpu ✓')
except Exception as e:
    print(f'Optional deps not installed ({e}). Keyword RAG fallback will be used.')

## Section 2 — Import Libraries
We import both the standard Python libraries (pandas, matplotlib, sklearn) and
the AgriGuard `ai/src` modules. The `sys.path` block makes the modules importable
when this notebook runs from the `ai/notebooks/` directory in Colab.

In [ ]:
import sys, os

# REPO_ROOT is set by Cell 0 (the clone cell above).
# If you are running locally (not Colab), set it manually:
#   REPO_ROOT = '/path/to/AGRIGUARD'
# Cell 0 already added REPO_ROOT to sys.path, so imports will work.

# Standard libraries
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

# AgriGuard AI modules
from ai.src import knowledge
from ai.src import classification
from ai.src import rag
from ai.src import vision
from ai.src.confidence import classify_confidence
from ai.src.severity import calculate_severity
from ai.src.alert_logic import needs_agronomist, calculate_distance_km, farmers_within_radius
from ai.src.multilingual import detect_language, get_message
from ai.src.outbreak_detection import detect_outbreak
from ai.src.crop_analysis import analyze_report, ussd_format, process_ussd_report
from ai.src.assistant import answer_farmer_question

print('All imports OK')
print(f'  pandas  {pd.__version__}')
print(f'  numpy   {np.__version__}')
print(f'  sklearn available: {classification.SKLEARN_AVAILABLE}')
print(f'  embeddings available: {rag.EMBEDDINGS_AVAILABLE}')

## Section 3 — Load Data
The prototype CSV datasets in `ai/data/` are generated from the canonical
knowledge base in `app/data/` (single source of truth). They contain:

| File | Rows | Description |
|------|------|-------------|
| `crops.csv` | 5 | Crop metadata (maize, beans, cassava, potato, tomato) |
| `crop_problems.csv` | 21 | Pests, diseases, nutrient deficiencies, environmental stress |
| `agricultural_knowledge.csv` | 31 | Flattened knowledge articles for RAG retrieval |
| `demo_reports.csv` | 16 | **PROTOTYPE** farmer reports (Kabarore cluster + isolated) |
| `demo_farmers.csv` | 9 | **PROTOTYPE** registered farmers with locations |

⚠️ **PROTOTYPE DATA — NOT REAL OUTBREAK DATA**

In [ ]:
# Load all datasets
crops      = knowledge.crops_df()
problems   = knowledge.problems_df()
articles   = knowledge.knowledge_df()
reports    = knowledge.reports_df()
farmers    = knowledge.farmers_df()

print(f'Crops:              {len(crops)} rows')
print(f'Problems:           {len(problems)} rows')
print(f'Knowledge articles: {len(articles)} rows')
print(f'Demo reports:       {len(reports)} rows  [PROTOTYPE]')
print(f'Demo farmers:       {len(farmers)} rows  [PROTOTYPE]')

print('\nCrop list:')
for _, row in crops.iterrows():
    print(f'  {row["id"]:15s} {row["name_en"]} ({row["scientific_name"]})')

## Section 4 — Explore the Data
Let's look at the structure of the knowledge base and the demo reports before
building any models.

In [ ]:
# 4a. Problem categories
print('=== Problem categories ===')
print(problems['category'].value_counts().to_string())

print('\n=== Problems by crop ===')
for crop_id in crops['id']:
    crop_problems = problems[problems['affected_crops'].str.contains(crop_id, na=False)]
    names = crop_problems['name_en'].tolist()
    print(f'  {crop_id:15s} ({len(names)}): {", ".join(names[:3])}{"..." if len(names)>3 else ""}')

print('\n=== Demo report severity distribution (PROTOTYPE) ===')
print(reports['severity'].value_counts().to_string())

print('\n=== Demo report communities (PROTOTYPE) ===')
print(reports['community'].value_counts().to_string())

In [ ]:
# 4b. Visualise problem categories
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Problem categories
cat_counts = problems['category'].value_counts()
colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']
axes[0].bar(cat_counts.index, cat_counts.values, color=colors[:len(cat_counts)])
axes[0].set_title('Problems by Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)

# Severity distribution in demo reports
sev_counts = reports['severity'].value_counts().reindex(['Low','Moderate','High','Critical']).fillna(0)
sev_colors = ['#2ecc71','#f39c12','#e67e22','#e74c3c']
axes[1].bar(sev_counts.index, sev_counts.values, color=sev_colors)
axes[1].set_title('Demo Report Severity\n(PROTOTYPE DATA)')
axes[1].set_ylabel('Count')

# Reports by community
comm_counts = reports['community'].value_counts()
axes[2].barh(comm_counts.index, comm_counts.values, color='#3498db')
axes[2].set_title('Demo Reports by Community\n(PROTOTYPE DATA)')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.savefig('04_data_exploration.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved.')

## Section 5 — Preprocess Data
Preprocessing for the text classifier:
- Each problem becomes multiple labelled training documents (name + symptoms + keyword bag).
- Tokenised with TF-IDF n-grams (unigrams + bigrams).
- No data from outside the knowledge base is used — 100% prototype.

In [ ]:
from ai.src.classification import _build_training_data

docs, labels, categories = _build_training_data()
print(f'Training documents: {len(docs)}')
print(f'Unique classes:     {len(set(labels))}')
print(f'Category breakdown:')
from collections import Counter
for cat, n in sorted(Counter(categories).items()):
    print(f'  {cat:25s} {n} docs')
print()
print('Example document:')
print(f'  label: {labels[0]}')
print(f'  text:  {docs[0][:120]}...')

## Section 6 — Crop / Problem Analysis
### Real Machine Learning Pipeline
```
Text (name + symptoms + keywords)
          ↓
TF-IDF vectoriser (unigrams + bigrams, sublinear TF)
          ↓
Logistic Regression classifier (class_weight='balanced')
          ↓
Predicted problem ID + probability (confidence)
```
This is a **real scikit-learn ML pipeline**, not a lookup table.

In [ ]:
if classification.SKLEARN_AVAILABLE:
    print('Training scikit-learn TF-IDF + Logistic Regression classifier...')
    eval_result = classification.train_classifier(test_size=0.25, random_state=42)

    print(f"  Training docs : {eval_result['n_train']}")
    print(f"  Test docs     : {eval_result['n_test']}")
    print(f"  Classes       : {eval_result['n_classes']}")
    print(f"  Accuracy      : {eval_result['accuracy']:.3f}")
    print(f"  Precision (macro): {eval_result['precision_macro']:.3f}")
    print(f"  Recall (macro)   : {eval_result['recall_macro']:.3f}")
    print(f"  F1 (macro)       : {eval_result['f1_macro']:.3f}")
    print(f"\n  NOTE: {eval_result['note']}")
else:
    print('scikit-learn not installed — using keyword fallback for classification.')

### Section 8 — Model Evaluation
Confusion matrix + per-class metrics. These are real evaluation results from the
held-out test split.  Because this is a **21-class prototype dataset** (not a
large field-collected dataset), the numbers are illustrative — do not interpret
them as production accuracy.

In [ ]:
if classification.SKLEARN_AVAILABLE and classification.is_ml_available():
    from sklearn.metrics import ConfusionMatrixDisplay
    import numpy as np

    labels_list = eval_result['labels']
    cm = np.array(eval_result['confusion_matrix'])

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    # Confusion matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_list)
    disp.plot(ax=axes[0], xticks_rotation='vertical', colorbar=False)
    axes[0].set_title('Confusion Matrix\n(21-class prototype — illustrative only)')
    axes[0].tick_params(axis='both', labelsize=7)

    # Per-class F1 scores
    report = eval_result['report']
    classes_f1 = {k: v['f1-score'] for k, v in report.items() if isinstance(v, dict) and k not in ('accuracy','macro avg','weighted avg')}
    sorted_items = sorted(classes_f1.items(), key=lambda x: x[1], reverse=True)
    class_names = [x[0] for x in sorted_items]
    f1_scores = [x[1] for x in sorted_items]
    bar_colors = ['#2ecc71' if f >= 0.5 else '#e74c3c' for f in f1_scores]
    axes[1].barh(class_names, f1_scores, color=bar_colors)
    axes[1].set_xlim(0, 1)
    axes[1].set_title('Per-class F1 Score')
    axes[1].set_xlabel('F1 Score')
    axes[1].tick_params(axis='y', labelsize=8)
    axes[1].axvline(x=eval_result['f1_macro'], color='navy', linestyle='--', label=f"Macro F1={eval_result['f1_macro']:.2f}")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('08_model_evaluation.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('ML classifier not available — skipping confusion matrix.')

## Section 7 — Confidence Calculation
Confidence and severity are **completely independent** concepts:
- **Confidence** = how certain the AI is about its identification.
- **Severity** = how serious the problem is for the crop.
A high-confidence result can still be high-severity, and vice versa.

In [ ]:
from ai.src.confidence import ConfidenceConfig

print('=== Confidence classification (default thresholds: high=0.80, medium=0.50) ===')
for score in [0.92, 0.65, 0.80, 0.50, 0.49, 0.38, 0.0]:
    r = classify_confidence(score)
    bar = '█' * int(r.score * 20)
    print(f'  {r.score:.2f}  {bar:20s}  {r.level:6s} ({r.tier})')

print('\n=== Custom thresholds demonstration ===')
custom = ConfidenceConfig(high=0.70, medium=0.40)
for score in [0.75, 0.55, 0.35]:
    r = classify_confidence(score, config=custom)
    print(f'  {score:.2f} -> {r.level} (custom thresholds: high={custom.high}, medium={custom.medium})')

## Section 8 — Severity Assessment
Severity is determined from the problem's data-driven baseline, then upgraded
(never downgraded) if a vision hint indicates wider damage. Unknown problems are
always assigned **High** severity so a human agronomist reviews them.

In [ ]:
from ai.src.severity import SEVERITY_LEVELS
from ai.src import knowledge as kb

print('Severity levels (ordered):', SEVERITY_LEVELS)
print()

# Show problem severities from the knowledge base
all_probs = kb.problems_as_dicts()
print(f'  {"Problem":35s} {"Category":20s} {"Severity"}')
print('-' * 72)
for p in all_probs:
    sev = p['severity_example']
    color = {'Low':'[LOW]','Moderate':'[MOD]','High':'[HIGH]','Critical':'[CRIT]'}.get(sev,'')
    print(f'  {p["name_en"]:35s} {p["category"]:20s} {color} {sev}')

print('\nKey principle: Confidence 0.94 + Severity High = AI is SURE this is serious.')
print('Do NOT assume High Confidence means Low Severity.')

## Section 9 — Agricultural Knowledge Retrieval (RAG)
```
Farmer Question
       ↓
  Embedding (all-MiniLM-L6-v2) OR keyword tokenisation
       ↓
  FAISS inner-product search OR keyword overlap scoring
       ↓
  Top-k knowledge articles
```
If `sentence-transformers` + `faiss-cpu` are installed, we use **real semantic
embeddings**. Otherwise the keyword fallback kicks in automatically.

In [ ]:
# Build the FAISS embeddings index if the dependencies are installed
if rag.EMBEDDINGS_AVAILABLE:
    print('Building sentence-transformers FAISS index...')
    n = rag.build_embeddings_index()
    print(f'  Indexed {n} knowledge articles with all-MiniLM-L6-v2 embeddings.')
else:
    print('sentence-transformers / faiss-cpu not installed — using keyword retrieval.')
    print(f'Retriever active: {rag.retriever_method()}')

# Demonstrate retrieval
print(f'\nActive retrieval method: {rag.retriever_method()}')
query = 'What should I do if my maize leaves have holes and there are insects?'
chunks = rag.retrieve_agricultural_knowledge(query, top_k=3, crop_id='maize')
print(f'\nQuery: "{query}"')
for i, c in enumerate(chunks):
    print(f"\n  [{i+1}] {c['title']}  (score={c['score']:.3f})")
    print(f"      {c['content'][:120]}...")
    print(f"      source: {c['source']}")

## Section 10 — AI Recommendation
Recommendations come from the knowledge base, not from the model fabricating
advice. Each recommendation is explicitly linked to the retrieved problem
record.

In [ ]:
from ai.src.recommendations import generate_recommendation

# 10a. Known problem
fall_armyworm = kb.get_problem_by_id('fall-armyworm')
recs_en = generate_recommendation('maize', fall_armyworm, lang='en', severity='Moderate')
recs_rw = generate_recommendation('maize', fall_armyworm, lang='rw', severity='Moderate')

print('=== Fall Armyworm — Recommendations (English) ===')
for i, r in enumerate(recs_en, 1):
    print(f'  {i}. {r}')

print('\n=== Fall Armyworm — Inama (Kinyarwanda) ===')
for i, r in enumerate(recs_rw, 1):
    print(f'  {i}. {r}')

# 10b. Unknown problem (falls back to crop prevention tips)
print('\n=== Unknown problem on Maize — fallback to prevention tips ===')
fallback_recs = generate_recommendation('maize', None, lang='en')
for i, r in enumerate(fallback_recs, 1):
    print(f'  {i}. {r}')

## Section 11 — Community Risk Analysis
Prototype risk score (0–100, transparent heuristic — NOT a probability):
```
Score = count_component (0-40)
      + severity_component (0-25)
      + confidence_component (0-15)
      + geo_concentration (0-10)
      + recency_bonus (0-10)
```

In [ ]:
# Load demo reports
all_reports = kb.reports_as_dicts()
print(f'Total demo reports loaded: {len(all_reports)} [PROTOTYPE]')
print()

# Show community groups
from collections import Counter
community_counts = Counter(r['community'] for r in all_reports)
problem_counts = Counter(r['problem'] for r in all_reports if r['problem'])

print('Reports by community:')
for comm, n in sorted(community_counts.items(), key=lambda x: -x[1]):
    print(f'  {comm:15s}: {n} report(s)')

print('\nReports by problem (top 5):')
for prob, n in problem_counts.most_common(5):
    print(f'  {prob:30s}: {n} report(s)')

## Section 12 — Outbreak Detection
The detector groups reports by (problem, crop), scores each group with the
transparent heuristic, and returns the most concerning cluster.

In [ ]:
# Scenario A: full dataset — Kabarore cluster
result = detect_outbreak(all_reports)
print('=== Outbreak Detection Result (PROTOTYPE) ===')
print(f'  Risk level     : {result.risk_level}')
print(f'  Risk score     : {result.risk_score}/100')
print(f'  Problem        : {result.problem}')
print(f'  Crop           : {result.crop}')
print(f'  Report count   : {result.report_count}')
print(f'  Affected area  : {result.affected_area}')
print(f'  Geo concentrated: {result.geo_concentrated}')
print(f'  Alert recommended: {result.alert_recommended}')
print(f'  Prototype: {result.is_prototype}  ← Always True for demo data')
print(f'  Disclaimer: {result.disclaimer}')

# Isolated reports only
isolated = [r for r in all_reports if r.get('community') != 'Kabarore']
result_isolated = detect_outbreak(isolated)
print(f'\nIsolated reports only → Risk: {result_isolated.risk_level} (expected LOW)')

In [ ]:
# Visualise risk score breakdown
risk_components = {
    'Report count\n(0-40)': min((result.report_count / 12) * 40, 40),
    'Avg severity\n(0-25)': (sum({'Low':1,'Moderate':2,'High':3,'Critical':4}.get(r.get('severity'),1)
                                 for r in [r for r in all_reports if r.get('community')=='Kabarore'])
                             / len([r for r in all_reports if r.get('community')=='Kabarore']) / 4) * 25,
    'Avg confidence\n(0-15)': (sum(r.get('confidence',0) for r in [r for r in all_reports if r.get('community')=='Kabarore'])
                               / len([r for r in all_reports if r.get('community')=='Kabarore'])) * 15,
    'Geo concentration\n(0-10)': 10 if result.geo_concentrated else 0,
    'Recency bonus\n(0-10)': min(10, result.risk_score - min((result.report_count/12)*40,40)),
}

fig, ax = plt.subplots(figsize=(10, 4))
colors_comp = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6']
bars = ax.bar(risk_components.keys(), risk_components.values(), color=colors_comp)
ax.set_ylim(0, 42)
ax.set_title(f'Risk Score Components — {result.affected_area}\nTotal score = {result.risk_score}/100 → {result.risk_level}\n(PROTOTYPE data)')
ax.set_ylabel('Component score')
for bar, val in zip(bars, risk_components.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val:.1f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('12_risk_components.png', dpi=120, bbox_inches='tight')
plt.show()

## Section 13 — 5 km Alert Logic
Only farmers who are **registered**, have a known location, **and** are within
the configured radius (default 5 km) receive a personalised prototype alert.
Farmers without coordinates are excluded — we never estimate unknown locations.

In [ ]:
all_farmers = kb.farmers_as_dicts()
print(f'Registered farmers with locations: {sum(1 for f in all_farmers if f["lat"] is not None)}/{len(all_farmers)}')

if result.affected_area_center:
    centre = result.affected_area_center
    alertable = farmers_within_radius(
        centre['lat'], centre['lng'], all_farmers, radius_km=5.0
    )
    print(f'\nOutbreak centre: lat={centre["lat"]:.4f}, lng={centre["lng"]:.4f} ({result.affected_area})')
    print(f'Alertable farmers within 5 km: {len(alertable)}')
    print()
    for f in alertable:
        print(f'  ✅  {f.farmer_id} {f.name or ""} — {f.distance_km:.2f} km — {f.community}')
    not_eligible = [f for f in all_farmers if not any(a.farmer_id == f['id'] for a in alertable)]
    for f in not_eligible:
        lat = f.get('lat')
        if lat is None:
            print(f'  ❌  {f["id"]} {f["name"]} — no location data')
        else:
            d = calculate_distance_km(centre['lat'], centre['lng'], lat, f['lng'])
            print(f'  ❌  {f["id"]} {f["name"]} — {d:.1f} km — outside 5 km radius')

In [ ]:
# Visualise farmer positions on a map (scatter plot)
if result.affected_area_center:
    fig, ax = plt.subplots(figsize=(8, 7))
    centre = result.affected_area_center

    # Draw 5 km circle (approximate in degrees)
    # 1 degree latitude ≈ 111 km, so 5 km ≈ 0.045 degrees
    circle = plt.Circle((centre['lng'], centre['lat']), 5/111,
                         color='red', fill=False, linestyle='--', linewidth=1.5, label='5 km alert radius')
    ax.add_patch(circle)

    # Plot farmers
    for f in all_farmers:
        lat, lng = f.get('lat'), f.get('lng')
        if lat is None:
            continue
        in_radius = any(a.farmer_id == f['id'] for a in alertable)
        color = '#e74c3c' if in_radius else '#95a5a6'
        marker = 's' if in_radius else 'o'
        ax.scatter(lng, lat, c=color, marker=marker, s=80, zorder=3)
        ax.annotate(f['id'], (lng + 0.001, lat + 0.001), fontsize=7)

    # Outbreak centre
    ax.scatter(centre['lng'], centre['lat'], c='black', marker='*', s=200, zorder=4, label='Outbreak centre')

    red_patch = mpatches.Patch(color='#e74c3c', label='Within 5 km (alert eligible)')
    grey_patch = mpatches.Patch(color='#95a5a6', label='Outside radius / no location')
    ax.legend(handles=[red_patch, grey_patch, plt.Line2D([],[],color='red',linestyle='--',label='5 km radius'),
                       plt.Line2D([],[],color='black',marker='*',linestyle='None',label='Outbreak centre')],
              fontsize=8, loc='upper right')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Farmer Alert Eligibility — 5 km Radius\n(PROTOTYPE DATA)')
    plt.tight_layout()
    plt.savefig('13_alert_map.png', dpi=120, bbox_inches='tight')
    plt.show()

## Section 14 — Agronomist Escalation
**Responsible AI principle**: the system MUST escalate to a human agronomist
when the AI is uncertain or the situation is serious. It never claims to replace
qualified agronomists.

In [ ]:
print('=== Escalation decision examples ===')
cases = [
    {'confidence': 0.38, 'severity': 'Moderate', 'category': 'Pest',    'farmerHelp': False},
    {'confidence': 0.92, 'severity': 'High',     'category': 'Disease', 'farmerHelp': False},
    {'confidence': 0.95, 'severity': 'Critical', 'category': 'Disease', 'farmerHelp': False},
    {'confidence': 0.45, 'severity': 'High',     'category': 'Unknown', 'farmerHelp': False},
    {'confidence': 0.92, 'severity': 'Moderate', 'category': 'Pest',    'farmerHelp': True},
    {'confidence': 0.92, 'severity': 'Moderate', 'category': 'Pest',    'farmerHelp': False},
]
print(f'  {"Conf":6s} {"Severity":10s} {"Category":20s} {"FarmerHelp":12s} {"Escalate?":12s} Reason')
print('-' * 90)
for c in cases:
    r = needs_agronomist(c['confidence'], c['severity'], c['category'], c['farmerHelp'])
    flag = '🔴 YES' if r.needs_agronomist else '🟢 NO '
    print(f'  {c["confidence"]:<6.2f} {c["severity"]:10s} {c["category"]:20s} {str(c["farmerHelp"]):12s} {flag}  {r.reason}')

## Section 15 — Multilingual Example
AgriGuard supports English and Kinyarwanda. Language is auto-detected from the
input; the caller can also pass it explicitly (recommended for short USSD inputs).

Honest limitation (Section 27): Kinyarwanda answer generation here is
template-based knowledge retrieval, not a generative LLM.

In [ ]:
print('=== Language Detection ===')
texts = [
    'My maize leaves are turning yellow, what should I do?',
    'Ibibabi by\'ibigori byanjye birahinduka umuhondo.',
    'What is causing holes in my maize leaves?',
    'Ibigori byanjye birimo gukangarwa n\'udukoko.',
]
for t in texts:
    lang = detect_language(t)
    print(f'  [{lang.upper()}] {t[:60]}...' if len(t) > 60 else f'  [{lang.upper()}] {t}')

print('\n=== Ask AgriGuard — English ===')
ans_en = answer_farmer_question(
    'How do I protect my maize from Fall Armyworm?', language='en'
)
print(f'Language: {ans_en.language}')
print(f'Confidence: {ans_en.confidence:.2f} | Needs agronomist: {ans_en.needs_agronomist}')
print(f'Retrieval method: {ans_en.retrieval_method}')
print(f'Answer:\n{ans_en.answer[:400]}...')

print('\n=== Kuribazo kw\'umuhinzi — Kinyarwanda ===')
ans_rw = answer_farmer_question(
    'Ibigori byanjye birimo gukangarwa n\'udukoko, mbwira inama.', language='rw'
)
print(f'Language: {ans_rw.language}')
print(f'Answer:\n{ans_rw.answer[:400]}...')

## Section 16 — End-to-End AgriGuard Demonstration
Four complete scenarios demonstrating the full pipeline.

---
### Scenario A — Smartphone: image + description → crop analysis

In [ ]:
print('=' * 60)
print('SCENARIO A — SMARTPHONE FLOW')
print('=' * 60)
print('Input:')
print('  Crop: Maize')
print('  Image: maize-armyworm.jpg  (simulated demo key)')
print('  Description: Holes in leaves, small insects near the whorl')
print('  Location: Kabarore')
print()

result_a = analyze_report(
    crop='Maize',
    description='The leaves of my maize have holes and I can see small insects near the whorl.',
    image='maize-armyworm.jpg',
    location={'lat': -1.479, 'lng': 30.419, 'community': 'Kabarore'},
    language='en',
)
d = result_a.to_dict()
print('Result:')
print(f'  Crop           : {d["crop"]} ({d["crop_id"]})')
print(f'  Category       : {d["category"]}')
print(f'  Possible problem: {d["possible_problem"]}')
print(f'  Confidence     : {d["confidence"]}  → {d["confidence_level"]}')
print(f'  Severity       : {d["severity"]}')
print(f'  Needs agronomist: {d["needs_agronomist"]}  ({d["escalation_reason"]})')
print(f'  Vision source  : {d["vision_source"]}  ← clearly labelled')
print(f'  ML source      : {d["source"]}')
print(f'  Recommendation:')
for rec in d['recommendation']:
    print(f'    • {rec}')

### Scenario B — Community outbreak + 5 km alert eligibility

In [ ]:
print('=' * 60)
print('SCENARIO B — COMMUNITY OUTBREAK + 5 KM ALERTS')
print('=' * 60)
all_reports = kb.reports_as_dicts()
all_farmers = kb.farmers_as_dicts()

outbreak = detect_outbreak(all_reports)
print(f'  Risk level     : {outbreak.risk_level}')
print(f'  Risk score     : {outbreak.risk_score}/100')
print(f'  Problem        : {outbreak.problem}')
print(f'  Crop           : {outbreak.crop}')
print(f'  Affected area  : {outbreak.affected_area}')
print(f'  PROTOTYPE      : {outbreak.is_prototype}')

if outbreak.alert_recommended and outbreak.affected_area_center:
    centre = outbreak.affected_area_center
    alertable = farmers_within_radius(centre['lat'], centre['lng'], all_farmers, radius_km=5.0)
    print(f'\n  Personalised alerts sent to {len(alertable)} farmer(s):')
    for f in alertable:
        print(f'    📱 {f.farmer_id} — {f.name} ({f.community}) — {f.distance_km:.2f} km')
else:
    print('  No personalised alert (below threshold).')

### Scenario C — Unknown problem → agronomist escalation

In [ ]:
print('=' * 60)
print('SCENARIO C — UNCERTAIN CASE → AGRONOMIST ESCALATION')
print('=' * 60)
result_c = analyze_report(
    crop='Tomato',
    description='Something strange is happening with my plants, I am not sure what it is.',
    language='en',
 use_ml=False,
)
d = result_c.to_dict()
print(f'  Category       : {d["category"]}')
print(f'  Possible problem: {d["possible_problem"]}')
print(f'  Confidence     : {d["confidence"]}  → {d["confidence_level"]}')
print(f'  Severity       : {d["severity"]}')
print(f'  Needs agronomist: {d["needs_agronomist"]}  ← ALWAYS escalates for Unknown/low confidence')
print(f'  Escalation reason: {d["escalation_reason"]}')
print()
print('USSD format (what the farmer sees on their feature phone):')
print(ussd_format(result_c))

### Scenario D — Kinyarwanda farmer question → grounded Kinyarwanda answer

In [ ]:
print('=' * 60)
print('SCENARIO D — KINYARWANDA ASSISTANT')
print('=' * 60)
question_rw = 'Ibibabi by\'ibigori byanjye birahinduka umuhondo, mbwira inama.'
print(f'Question: {question_rw}')
print()
ans = answer_farmer_question(question_rw, language='rw', crop_id='maize')
print(f'Detected language : {ans.language}')
print(f'Retrieval method  : {ans.retrieval_method}')
print(f'Confidence        : {ans.confidence:.2f}')
print(f'Needs agronomist  : {ans.needs_agronomist}')
print(f'Sources:')
for s in ans.sources:
    print(f'  • {s["title"]} ({s["source"]})')
print(f'\nAnswer:\n{ans.answer}')

### Section 16 continued — Data visualisations

In [ ]:
# Final summary dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Reports by crop
crop_counts = reports['crop'].value_counts()
axes[0, 0].bar(crop_counts.index, crop_counts.values,
               color=['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6'][:len(crop_counts)])
axes[0, 0].set_title('Reports by Crop (PROTOTYPE)')
axes[0, 0].set_ylabel('Count')

# 2. Category distribution in problems
cat_c = problems['category'].value_counts()
axes[0, 1].pie(cat_c.values, labels=cat_c.index, autopct='%1.0f%%',
               colors=['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6'][:len(cat_c)])
axes[0, 1].set_title('Problem Categories in Knowledge Base')

# 3. Confidence tiers demonstrated
scores_demo = [0.92, 0.65, 0.38, 0.80, 0.50, 0.49, 0.78, 0.44, 0.91]
tier_colors = ['#2ecc71' if classify_confidence(s).tier == 'high'
               else '#f39c12' if classify_confidence(s).tier == 'medium'
               else '#e74c3c' for s in scores_demo]
bars = axes[1, 0].bar(range(len(scores_demo)), scores_demo, color=tier_colors)
axes[1, 0].axhline(0.80, color='green', linestyle='--', alpha=0.7, label='High threshold (0.80)')
axes[1, 0].axhline(0.50, color='orange', linestyle='--', alpha=0.7, label='Medium threshold (0.50)')
axes[1, 0].set_title('Confidence Tier Classification')
axes[1, 0].set_xlabel('Example index')
axes[1, 0].set_ylabel('Confidence score')
axes[1, 0].legend(fontsize=8)
for bar, s in zip(bars, scores_demo):
    tier = classify_confidence(s).level[0]
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, s + 0.01, tier, ha='center', fontsize=9, fontweight='bold')

# 4. Risk level comparison
risk_scenarios = ['Kabarore\ncluster (12)', 'Isolated\nreports (4)', 'Empty\nlist']
risk_scores_demo = [outbreak.risk_score, result_isolated.risk_score, 0]
risk_colors_demo = ['#e74c3c','#2ecc71','#2ecc71']
axes[1, 1].bar(risk_scenarios, risk_scores_demo, color=risk_colors_demo)
axes[1, 1].axhline(50, color='orange', linestyle='--', label='HIGH threshold (50)')
axes[1, 1].axhline(70, color='red', linestyle='--', label='CRITICAL threshold (70)')
axes[1, 1].set_title('Prototype Risk Score Comparison')
axes[1, 1].set_ylabel('Risk score (0-100)')
axes[1, 1].set_ylim(0, 105)
axes[1, 1].legend(fontsize=8)
for bar, score, label in zip(axes[1,1].patches, risk_scores_demo,
                              [outbreak.risk_level, result_isolated.risk_level, 'LOW']):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, score + 1, label, ha='center', fontsize=9, fontweight='bold')

plt.suptitle('AgriGuard AI Summary Dashboard (PROTOTYPE DATA)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('16_summary_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print('All visualisations saved.')

## ✅ Demo Complete

| Scenario | Input | AI Output |
|----------|-------|-----------|
| A — Smartphone | Maize + image + description | Fall Armyworm, High confidence, Moderate severity |
| B — Community | 16 prototype reports | CRITICAL/HIGH risk, Kabarore, 5 km alerts sent |
| C — Uncertain | Unknown tomato problem | Unknown category, Low confidence → escalate |
| D — Kinyarwanda | Kinyarwanda question | Grounded Kinyarwanda agricultural answer |

### What is simulated vs real?
| Component | Status |
|-----------|--------|
| TF-IDF + Logistic Regression classifier | **Real ML** (scikit-learn) |
| RAG embeddings (sentence-transformers + FAISS) | **Real ML** (when installed) |
| RAG keyword fallback | Deterministic heuristic |
| Vision analysis | **Simulated demo** (`source='simulated-demo'`) — clearly labelled |
| Prototype knowledge base | Demo data from public agricultural literature |
| Farmer reports/locations | **PROTOTYPE** — NOT real outbreak data |
| Risk scoring | Transparent heuristic — NOT a validated forecasting model |

### Model limitations (Section 27)
- Dataset may not represent all Rwandan crops and field conditions.
- Image quality affects predictions.
- Similar symptoms can belong to different problems.
- Kinyarwanda AI performance must be validated.
- Prototype outbreak detection is NOT a scientifically validated model.
- AI recommendations do NOT replace qualified agronomists.
- Real deployment requires local field validation.

In [ ]:
# USSD integration check (Section 17) — verify same AI, different format
print('=== USSD Integration Test ===')
ussd_response = process_ussd_report(
    farmer_id='F003',
    crop='Maize',
    problem_description='Small insects eating the maize leaves near the top',
    location='Kabarore',
    language='en',
)
print(f'USSD response for farmer {ussd_response["farmer_id"]}:')
print(ussd_response['ussd_response'])

---
## Section 17 — Voice AI: Speech Input and Spoken Output

AgriGuard supports a **voice-based AI interaction** for farmers — essential for
IVR (phone call) channels and low-literacy users.

### Complete pipeline
```
Farmer speaks  →  Audio file
      ↓  Speech-to-Text (Whisper — offline, free, no API key)
Kinyarwanda / English text
      ↓  Same agricultural AI already built (RAG + escalation)
AI-generated answer text
      ↓  Text-to-Speech (gTTS — free, requires internet)
Spoken audio response  →  Farmer hears the answer
```

### Channels — same AI core serves all three:
| Channel | Input | Output |
|---------|-------|--------|
| Smartphone | photo + text + voice | text + audio |
| USSD (feature phone) | text menu | short plain text |
| IVR (phone call) | farmer speaks | AI speaks back |

### Kinyarwanda limitations (honest documentation)
- **STT**: Whisper includes Kinyarwanda but quality in Rwandan farm conditions is unvalidated.
- **TTS**: No native Kinyarwanda voice exists. gTTS reads Kinyarwanda text with an English voice.
  Every result carries a `rw_tts_note` so callers are never misled.

### 17-A — Setup: clone repo, install voice deps, import everything

**Run this single cell first.** It handles everything:
1. Makes sure the repo is cloned and on `sys.path`
2. Installs `gTTS`, `pyttsx3`, `openai-whisper`, `ffmpeg`
3. Imports all voice modules

If a package fails to install, the pipeline falls back to the clearly-labelled
Stub backends automatically — the demo always runs.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SECTION 17-A  — Self-contained setup cell for Voice AI
# Run this cell BEFORE any other Section 17 cell.
# ═══════════════════════════════════════════════════════════════════
import os, sys, subprocess, importlib, wave, struct

# ── Step 1: Clone or hard-reset to latest main ───────────────────────
REPO_URL  = 'https://github.com/Klab-TekHer-Cohort2/AGRIGUARD.git'
CLONE_DIR = '/content/AGRIGUARD'

if not os.path.exists(CLONE_DIR):
    print('Cloning repository...')
    ret = os.system(f'git clone --depth 1 {REPO_URL} {CLONE_DIR}')
    print('Clone done.' if ret == 0 else f'Clone failed (exit {ret})')
else:
    print('Repo already present — fetching + hard-reset to latest main...')
    os.system(f'git -C {CLONE_DIR} fetch origin main -q')
    os.system(f'git -C {CLONE_DIR} reset --hard origin/main -q')
    print('Reset to origin/main done.')

# ── Step 2: Verify critical files exist before importing ────────────────
REQUIRED = [
    'ai/__init__.py',
    'ai/src/__init__.py',
    'ai/src/stt.py',
    'ai/src/tts.py',
    'ai/src/voice_assistant.py',
    'ai/data/crops.csv',
]
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(CLONE_DIR, f))]
if missing:
    print('\n❌ These files are missing after clone/reset:')
    for m in missing:
        print(f'   {m}')
    print('\nFix: re-run this cell, or delete /content/AGRIGUARD and run again.')
    raise FileNotFoundError(f'Missing repo files: {missing}')
print('All required files found ✓')

# ── Step 3: Add repo root to sys.path ───────────────────────────────────
if CLONE_DIR not in sys.path:
    sys.path.insert(0, CLONE_DIR)
os.environ['AGRIGUARD_DATA_DIR'] = os.path.join(CLONE_DIR, 'ai', 'data')
print(f'sys.path[0] = {sys.path[0]}')

# ── Step 4: Install voice dependencies ──────────────────────────────────
def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

print('\nInstalling gTTS + pyttsx3 + nest-asyncio...')
try:
    _pip('gTTS', 'pyttsx3', 'nest-asyncio')
    print('  gTTS + pyttsx3 ✓')
except Exception as e:
    print(f'  TTS install skipped ({e}) — Stub TTS will be used')

print('Installing openai-whisper + ffmpeg...')
try:
    _pip('openai-whisper')
    os.system('apt-get install -y ffmpeg -qq')
    print('  openai-whisper + ffmpeg ✓')
except Exception as e:
    print(f'  Whisper install skipped ({e}) — Stub STT will be used')

# ── Step 5: Clear stale cached modules + import fresh ───────────────────
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('ai.'):
        del sys.modules[mod_name]

from ai.src.stt import speech_to_text, WHISPER_AVAILABLE, load_whisper
from ai.src.tts import text_to_speech, GTTS_AVAILABLE, PYTTSX3_AVAILABLE
from ai.src.voice_assistant import voice_farmer_assistant, voice_crop_report
from ai.src import knowledge
from ai.src.outbreak_detection import detect_outbreak
from ai.src.alert_logic import farmers_within_radius
from ai.src.crop_analysis import analyze_report, ussd_format
from ai.src.confidence import classify_confidence

print('\n✓ All voice modules imported')
print(f'  Whisper (real STT) : {WHISPER_AVAILABLE}')
print(f'  gTTS    (real TTS) : {GTTS_AVAILABLE}')
print(f'  pyttsx3 (real TTS) : {PYTTSX3_AVAILABLE}')
print(f'  Fallback Stub      : always available')

# ── Step 6: Load Whisper model ───────────────────────────────────────────
if WHISPER_AVAILABLE:
    print('\nLoading Whisper base model (~140 MB, cached after first run)...')
    try:
        load_whisper(model_name='base', device='cpu')
        print('  Whisper base model loaded ✓')
        print('  Kinyarwanda: included in training (field quality unvalidated)')
    except Exception as e:
        print(f'  Whisper load failed: {e} — Stub STT will be used')
        WHISPER_AVAILABLE = False
else:
    print('\nWhisper not installed — StubSTT will be used (source=stub-simulated)')

# ── Step 7: Create demo audio placeholder files ──────────────────────────
def _silent_wav(path, secs=1, rate=16000):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    n = int(rate * secs)
    with wave.open(path, 'w') as wf:
        wf.setnchannels(1); wf.setsampwidth(2)
        wf.setframerate(rate)
        wf.writeframes(struct.pack(f'<{n}h', *([0]*n)))
    return path

DEMO_EN  = _silent_wav('/tmp/farmer_question_en.wav')
DEMO_RW  = _silent_wav('/tmp/farmer_armyworm_rw.wav')
DEMO_UNK = _silent_wav('/tmp/farmer_unknown_problem.wav')
DEMO_IMG = _silent_wav('/tmp/farmer_armyworm_voice.wav')

print('\nDemo audio placeholders ready:')
for p in [DEMO_EN, DEMO_RW, DEMO_UNK, DEMO_IMG]:
    print(f'  {p}')
print('\n💡 To use your own audio:')
print('   from google.colab import files')
print('   uploaded = files.upload()          # pick your WAV/MP3')
print('   DEMO_EN = list(uploaded.keys())[0] # then re-run the demo cells')
print('\n✅ Setup complete — run the demo cells below.')


### 17-B — Demo: English voice question → spoken agricultural answer

In [ ]:
print('=' * 60)
print('VOICE DEMO — ENGLISH')
print('=' * 60)

result_en = voice_farmer_assistant(
    audio_path=DEMO_EN,
    language='en',
    tts_output_path='/tmp/response_en.mp3',
    use_real_stt=WHISPER_AVAILABLE,
    tts_prefer='gtts',
)

print(f"Recognised speech : {result_en['transcribed_text']}")
print(f"Language          : {result_en['language']}")
print(f"STT source        : {result_en['stt_source']}")
print(f"AI confidence     : {result_en['confidence']}")
print(f"Needs agronomist  : {result_en['needs_agronomist']}")
print(f"TTS source        : {result_en['tts_source']}")
print(f"Audio saved to    : {result_en['audio_path']}")
print(f"\nAI response:")
print(result_en['answer_text'][:500])

# ── Play audio in Colab ──────────────────────────────────────────────────
from IPython.display import Audio, display
audio_path = result_en['audio_path']
if os.path.exists(audio_path) and os.path.getsize(audio_path) > 200:
    print('\n🔊 Playing spoken response:')
    display(Audio(audio_path, autoplay=False))
else:
    print('\n(Silent placeholder — gTTS not installed or no internet. Install gTTS for real speech.)')

### 17-C — Demo: Kinyarwanda voice question → spoken answer

⚠️ **Kinyarwanda TTS**: No native voice — an English voice reads the Kinyarwanda text.
The `rw_tts_note` field always flags this.

In [ ]:
print('=' * 60)
print('VOICE DEMO — KINYARWANDA')
print('=' * 60)

result_rw = voice_farmer_assistant(
    audio_path=DEMO_RW,
    language='rw',
    tts_output_path='/tmp/response_rw.mp3',
    use_real_stt=WHISPER_AVAILABLE,
    tts_prefer='gtts',
)

print(f"Recognised speech : {result_rw['transcribed_text']}")
print(f"Language          : {result_rw['language']}")
print(f"AI confidence     : {result_rw['confidence']}")
print(f"Needs agronomist  : {result_rw['needs_agronomist']}")
print(f"Audio saved to    : {result_rw['audio_path']}")
print(f"\n⚠️  TTS limitation:")
print(f"   {result_rw['rw_tts_note']}")
print(f"\nAI response (Kinyarwanda):")
print(result_rw['answer_text'][:500])

from IPython.display import Audio, display
audio_path = result_rw['audio_path']
if os.path.exists(audio_path) and os.path.getsize(audio_path) > 200:
    print('\n🔊 Playing spoken response (English voice reading Kinyarwanda):')
    display(Audio(audio_path, autoplay=False))
else:
    print('\n(Silent placeholder — install gTTS for real speech output)')

### 17-D — Demo: Unknown problem → voice escalation message

When confidence is low or the problem is unrecognised, the AI produces a
spoken escalation message directing the farmer to an agronomist.

In [ ]:
print('=' * 60)
print('VOICE DEMO — UNKNOWN PROBLEM → ESCALATION')
print('=' * 60)

result_unk = voice_farmer_assistant(
    audio_path=DEMO_UNK,
    language='en',
    tts_output_path='/tmp/response_unknown.mp3',
    use_real_stt=False,       # Stub returns 'something strange' transcription
    tts_prefer='gtts',
)

print(f"Recognised speech  : {result_unk['transcribed_text']}")
print(f"Language           : {result_unk['language']}")
print(f"AI confidence      : {result_unk['confidence']}")
print(f"Needs agronomist   : {result_unk['needs_agronomist']}")
print(f"Escalation reason  : {result_unk['escalation_reason']}")
print(f"\nFarmer hears (spoken):")
print(result_unk['answer_text'][:500])

from IPython.display import Audio, display
audio_path = result_unk['audio_path']
if os.path.exists(audio_path) and os.path.getsize(audio_path) > 200:
    print('\n🔊 Playing escalation message:')
    display(Audio(audio_path, autoplay=False))
else:
    print('\n(Silent placeholder — install gTTS for spoken escalation message)')

### 17-E — Demo: Voice description + crop image → spoken recommendation

Combines **photo** + **voice** — the pipeline runs vision analysis + STT +
classification + TTS and speaks the recommendation back to the farmer.
This is the IVR + smartphone combined flow.

In [ ]:
print('=' * 60)
print('VOICE + IMAGE DEMO  (IVR + Smartphone combined)')
print('=' * 60)
print('Input:')
print('  Audio : farmer_armyworm_voice.wav')
print('  Image : maize-armyworm.jpg  (simulated demo key)')
print('  Crop  : Maize')
print()

result_vi = voice_crop_report(
    audio_path=DEMO_IMG,
    crop='Maize',
    image='maize-armyworm.jpg',
    location={'lat': -1.479, 'lng': 30.419, 'community': 'Kabarore'},
    language='en',
    tts_output_path='/tmp/response_voice_image.mp3',
    use_real_stt=False,
    tts_prefer='gtts',
    vision_mode='simulated',
)

ca = result_vi['crop_analysis']
print(f"Transcribed voice  : {result_vi['transcribed_text']}")
print(f"Language           : {result_vi['language']}")
print(f"Vision source      : {result_vi['vision_source']}  ← always labelled")
print(f"STT source         : {result_vi['stt_source']}")
print(f"Crop               : {ca['crop']} ({ca['crop_id']})")
print(f"Category           : {ca['category']}")
print(f"Possible problem   : {ca['possible_problem']}")
print(f"Confidence         : {ca['confidence']}  → {ca['confidence_level']}")
print(f"Severity           : {ca['severity']}")
print(f"Needs agronomist   : {result_vi['needs_agronomist']}")
print(f"Audio saved to     : {result_vi['audio_path']}")
print(f"\nFarmer hears (spoken recommendation):")
print(result_vi['answer_text'])

from IPython.display import Audio, display
audio_path = result_vi['audio_path']
if os.path.exists(audio_path) and os.path.getsize(audio_path) > 200:
    print('\n🔊 Playing spoken recommendation:')
    display(Audio(audio_path, autoplay=False))
else:
    print('\n(Silent placeholder — install gTTS for spoken recommendation)')

### 17-F — Full AgriGuard AI Architecture

```
              AGRIGUARD AI / DATA LAYER  (Python)
                            │
          ┌─────────────────┼─────────────────┐
          │                 │                 │
     IMAGE INPUT        TEXT INPUT       VOICE INPUT
          │                 │                 │
          ↓                 ↓                 ↓
   Computer Vision     Language AI     Speech-to-Text
   (vision.py)        (multilingual)   (stt.py — Whisper)
          │                 │                 │
          └─────────────────┼─────────────────┘
                            ↓
                  Agricultural AI Core
              (crop_analysis.py / assistant.py)
                            ↓
              Knowledge Retrieval — RAG
         (rag.py: sentence-transformers + FAISS)
                            ↓
                  AI Response Generation
                            ↓
          ┌─────────────────┼──────────────────┐
          │                 │                  │
        Text          Recommendation     Risk Analysis
          │                                   │
          ↓                                   ↓
    Text-to-Speech                   Outbreak Detection
   (tts.py — gTTS)                 + 5 km Alert Logic
          │
          ↓
    Spoken Response
    (IVR / Smartphone)
```

**All three channels use the same Python AI core:**
```
Smartphone  → photo + text + voice → AI → text + spoken response
USSD        → text menu            → AI → short plain-text response
IVR / Voice → farmer speaks        → STT → AI → TTS → farmer hears answer
```